# Huấn luyện giai đoạn 2 — levelA

Giai đoạn này khởi tạo mạng từ checkpoint của baseline L0 (giai đoạn 1), rồi fine-tune với:
- Curriculum L0/L1/L2 (trọng số 10/30/60)
- Hỗn hợp đối thủ: 30% ngẫu nhiên + 70% phòng thủ heuristic
- Đầu vào 6 kênh (có attack planes)
- D4 symmetry augmentation
- Phần thưởng đồng nhất cho các kết cục xấu (−1)

**Yêu cầu trước khi chạy:**
1. File `kqk.ipynb` ở `/content/drive/MyDrive/KHDL/checkpoints/` phải là bản mới (có `defender_move`, `N_PLANES_ATTACK`, `augment_obs`).
2. File `baseline_L0_best.pt` đã có ở `/content/drive/MyDrive/KHDL_baseline/checkpoints/` (từ huấn luyện giai đoạn 1).


In [ ]:
!pip install -q chess
!jupyter nbconvert --to python "/content/drive/MyDrive/KHDL/checkpoints/kqk.ipynb"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/KHDL/checkpoints/kqk.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equi

In [ ]:
import os, sys, json, time, random, importlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/KHDL"
CKPT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
KQK_IPYNB = os.path.join(CKPT_DIR, "kqk_v2.ipynb")
KQK_FILE = os.path.join(CKPT_DIR, "kqk_v2.py")


if not os.path.exists(KQK_IPYNB):
    raise FileNotFoundError(f"Khong tim thay: {KQK_IPYNB}")
get_ipython().system('jupyter nbconvert --to python "/content/drive/MyDrive/KHDL/checkpoints/kqk.ipynb"')
if not os.path.exists(KQK_FILE):
    raise FileNotFoundError(f"Chuyen doi that bai: {KQK_FILE}")

for p in (PROJECT_DIR, CKPT_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
sys.modules.pop("kqk", None)
import kqk
importlib.reload(kqk)
print("Da nap module:", kqk.__file__)

required = ["N_PLANES_ATTACK", "N_SYM", "augment_obs", "augment_action",
            "augment_mask", "defender_move"]
missing = [r for r in required if not hasattr(kqk, r)]
if missing:
    raise ImportError(f"kqk thieu cac ham: {missing}. Can cap nhat kqk ban moi.")
print(f"kqk hop le, N_PLANES_ATTACK={kqk.N_PLANES_ATTACK}, N_SYM={kqk.N_SYM}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiet bi:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
[NbConvertApp] Converting notebook /content/drive/MyDrive/KHDL/checkpoints/kqk.ipynb to python
[NbConvertApp] Writing 15402 bytes to /content/drive/MyDrive/KHDL/checkpoints/kqk.py
292 e1e5 21
. . . . . . Q .
. . . k . . . .
. . . . . . . .
. K . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Potential của vị trí ban đầu: 0.286
292 e1e5 21
. . . . . . Q .
. . . k . . . .
. . . . . . . .
. K . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Potential của vị trí ban đầu: 0.286
Da nap module: /content/drive/MyDrive/KHDL/checkpoints/kqk.py
kqk hop le, N_PLANES_ATTACK=6, N_SYM=8
Thiet bi: cpu


In [ ]:
from collections import Counter
import pandas as pd
HOLDOUT_SEED = 50_000

BASELINE_CKPT = "/content/drive/MyDrive/KHDL_baseline/checkpoints/baseline_L0_best.pt"
assert os.path.exists(BASELINE_CKPT), (
    f"Khong tim thay {BASELINE_CKPT}. Hay copy file sang hoac sua duong dan."
)

CONFIG = dict(
    seed=0,
    max_moves=50,
    num_episodes=10_000,
    train_levels=[0, 1, 2],
    level_weights=[0.1, 0.3, 0.6],
    opponent_mix=dict(random=0.3, defender=0.7),
    attack_planes=True,
    symmetry_aug=True,
    init_from=BASELINE_CKPT,
    gamma=0.95, lr=1e-4, batch_size=64, buffer_size=50_000, learn_start=1_000,
    tau=0.005, grad_clip=10.0, double_dqn=True,
    eps_start=0.3, eps_end=0.05, eps_decay_steps=40_000,
    explore_mate_in_one=True, train_freq=1,
    win_reward=5.0, lose_queen_reward=-1.0, stalemate_reward=-1.0,
    timeout_reward=-1.0, step_penalty=-0.01, shaping_weight=0.3,
    eval_every=1000, eval_games=200,
    eval_level=2, extra_eval_level=1, eval_seed=10_001,
    resume=True,
)


In [ ]:
class QNet(nn.Module):
    def __init__(self, in_ch=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.flatten = nn.Flatten()
        feat_dim = 64 * 8 * 8
        self.value = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(),
                                   nn.Linear(256, 1))
        self.advantage = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(),
                                       nn.Linear(256, kqk.N_ACTIONS))

    def forward(self, x):
        feat = self.flatten(self.conv(x))
        v = self.value(feat)
        a = self.advantage(feat)
        return v + a - a.mean(dim=1, keepdim=True)

print(sum(p.numel() for p in QNet(kqk.N_PLANES_ATTACK).parameters()), "tham so")


3170849 tham so


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, n_planes=3, augment=False):
        self.capacity = capacity
        self.augment = augment
        self.s = np.zeros((capacity, n_planes, 8, 8), dtype=np.uint8)
        self.s2 = np.zeros((capacity, n_planes, 8, 8), dtype=np.uint8)
        self.a = np.zeros(capacity, dtype=np.int64)
        self.r = np.zeros(capacity, dtype=np.float32)
        self.m2 = np.zeros((capacity, kqk.N_ACTIONS), dtype=bool)
        self.d = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.size = 0

    def push(self, s, a, r, s2, m2, d):
        i = self.pos
        self.s[i] = s; self.s2[i] = s2
        self.a[i] = a; self.r[i] = r
        self.m2[i] = m2; self.d[i] = d
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        s, s2 = self.s[idx], self.s2[idx]
        a, m2 = self.a[idx], self.m2[idx]
        if self.augment:
            k = np.random.randint(0, kqk.N_SYM, size=batch_size)
            s, s2 = kqk.augment_obs(s, k), kqk.augment_obs(s2, k)
            a = kqk.augment_action(a, k)
            m2 = kqk.augment_mask(m2, k)
        s = torch.from_numpy(np.ascontiguousarray(s)).float().to(device)
        s2 = torch.from_numpy(np.ascontiguousarray(s2)).float().to(device)
        a = torch.from_numpy(np.ascontiguousarray(a)).to(device)
        r = torch.from_numpy(self.r[idx]).to(device)
        m2 = torch.from_numpy(np.ascontiguousarray(m2)).to(device)
        d = torch.from_numpy(self.d[idx]).to(device)
        return s, a, r, s2, m2, d

    def __len__(self):
        return self.size


In [ ]:
@torch.no_grad()
def greedy_action(net, obs, mask):
    x = torch.from_numpy(obs).unsqueeze(0).to(device)
    q = net(x)[0]
    q = q.masked_fill(~torch.from_numpy(mask).to(device), float("-inf"))
    return int(q.argmax().item())

def mate_in_one_action(board):
    for mv in list(board.legal_moves):
        board.push(mv)
        mate = board.is_checkmate()
        board.pop()
        if mate:
            return kqk.move_to_index(mv)
    return None

def select_action(net, env, obs, mask, eps, rng, config):
    if rng.random() < eps:
        if config["explore_mate_in_one"] and rng.random() < 0.5:
            a = mate_in_one_action(env.board)
            if a is not None:
                return a
        legal = np.flatnonzero(mask)
        return int(legal[rng.randrange(len(legal))])
    return greedy_action(net, obs, mask)

def optimize(policy_net, target_net, opt, buffer, cfg):
    s, a, r, s2, m2, d = buffer.sample(cfg["batch_size"])
    q = policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        if cfg["double_dqn"]:
            a2 = policy_net(s2).masked_fill(~m2, float("-inf")).argmax(dim=1, keepdim=True)
            q2 = target_net(s2).gather(1, a2).squeeze(1)
        else:
            q2 = target_net(s2).masked_fill(~m2, float("-inf")).max(dim=1).values
        q2 = torch.where(d > 0.5, torch.zeros_like(q2), q2)
        target = r + cfg["gamma"] * q2
    loss = F.smooth_l1_loss(q, target)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), cfg["grad_clip"])
    opt.step()
    return loss.item()

def soft_update(target_net, policy_net, tau):
    for tp, pp in zip(target_net.parameters(), policy_net.parameters()):
        tp.data.copy_(tau * pp.data + (1.0 - tau) * tp.data)

def make_agent_policy(net):
    net.eval()
    def agent_policy(env, rng):
        obs = kqk.board_to_tensor(env.board, env.attack)
        mask = kqk.legal_mask(env.board)
        return greedy_action(net, obs, mask)
    return agent_policy

def evaluate(net, level, n_games, seed, cfg, opponent="defender"):
    env_e = kqk.KQKEnv(max_moves=cfg["max_moves"], shaping_weight=0.0,
                       attack=cfg["attack_planes"], opponent=opponent)
    stats = kqk.play_games(env_e, make_agent_policy(net), n_games=n_games,
                           level=level, seed=seed)
    net.train()
    return stats

def load_init_weights(net, state_dict, new_in_ch):
    sd = dict(state_dict)
    key = "conv.0.weight"
    old = sd[key]
    old_in = old.shape[1]
    if old_in != new_in_ch:
        new = torch.zeros(old.shape[0], new_in_ch, old.shape[2], old.shape[3],
                          dtype=old.dtype, device=old.device)
        new[:, :old_in] = old
        sd[key] = new
        print(f"Mo rong conv1 tu {old_in} sang {new_in_ch} kenh "
              f"(cac kenh them khoi tao bang 0)")
    net.load_state_dict(sd)


In [ ]:
def sanity_check():
    cfg = dict(CONFIG)
    n_planes = kqk.N_PLANES_ATTACK if cfg["attack_planes"] else kqk.N_PLANES_BASE
    env = kqk.KQKEnv(max_moves=cfg["max_moves"], level=0, seed=0,
                     win_reward=cfg["win_reward"],
                     lose_queen_reward=cfg["lose_queen_reward"],
                     stalemate_reward=cfg["stalemate_reward"],
                     timeout_reward=cfg["timeout_reward"],
                     shaping_weight=cfg["shaping_weight"],
                     shaping_gamma=cfg["gamma"],
                     attack=cfg["attack_planes"])
    buf = ReplayBuffer(500, n_planes, augment=cfg["symmetry_aug"])
    net = QNet(n_planes).to(device)
    tgt = QNet(n_planes).to(device)
    tgt.load_state_dict(net.state_dict())
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"])
    rng = random.Random(0)

    out = net(torch.zeros(2, n_planes, 8, 8, device=device))
    assert out.shape == (2, kqk.N_ACTIONS)

    while len(buf) < 300:
        obs, info = env.reset()
        mask = info["legal_mask"]
        done = False
        while not done and len(buf) < 300:
            a = select_action(net, env, obs, mask, 1.0, rng, cfg)
            assert mask[a]
            obs2, r, term, trunc, info = env.step(a)
            buf.push(obs, a, r, obs2, info["legal_mask"], float(term or trunc))
            obs, mask = obs2, info["legal_mask"]
            done = term or trunc

    for _ in range(5):
        loss = optimize(net, tgt, opt, buf, cfg)
        assert np.isfinite(loss)
    soft_update(tgt, net, cfg["tau"])
    print("Kiem tra nhanh: OK. Loss mau =", round(loss, 4))

sanity_check()


Kiem tra nhanh: OK. Loss mau = 0.0219


In [ ]:
env_b = kqk.KQKEnv(max_moves=CONFIG["max_moves"], shaping_weight=0.0)
baseline = {}
for level in sorted({CONFIG["eval_level"], CONFIG["extra_eval_level"]}):
    for pol in (kqk.random_policy, kqk.greedy_policy):
        s = kqk.play_games(env_b, pol, n_games=CONFIG["eval_games"],
                           level=level, seed=CONFIG["eval_seed"])
        baseline[(pol.__name__, level)] = s
        print(f"{pol.__name__:14s} level={level}  chieu het={s['checkmate_rate']:.1%}  "
              f"mat hau={s['queen_loss_rate']:.1%}  stalemate={s['stalemate_rate']:.1%}")


random_policy  level=1  chieu het=1.5%  mat hau=73.0%  stalemate=9.0%
greedy_policy  level=1  chieu het=13.0%  mat hau=74.5%  stalemate=5.0%
random_policy  level=2  chieu het=1.0%  mat hau=77.5%  stalemate=8.0%
greedy_policy  level=2  chieu het=8.5%  mat hau=75.0%  stalemate=6.0%


In [ ]:
def make_env(cfg, seed):
    return kqk.KQKEnv(
        max_moves=cfg["max_moves"], seed=seed,
        win_reward=cfg["win_reward"], lose_queen_reward=cfg["lose_queen_reward"],
        stalemate_reward=cfg["stalemate_reward"], timeout_reward=cfg["timeout_reward"],
        step_penalty=cfg["step_penalty"], shaping_weight=cfg["shaping_weight"],
        shaping_gamma=cfg["gamma"], attack=cfg["attack_planes"],
    )

def train(cfg, tag="levelA"):
    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    rng = random.Random(cfg["seed"])
    mix = random.Random(cfg["seed"] + 1)
    n_planes = kqk.N_PLANES_ATTACK if cfg["attack_planes"] else kqk.N_PLANES_BASE

    env = make_env(cfg, cfg["seed"])
    policy_net = QNet(n_planes).to(device)
    target_net = QNet(n_planes).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    opt = torch.optim.Adam(policy_net.parameters(), lr=cfg["lr"])
    buffer = ReplayBuffer(cfg["buffer_size"], n_planes, augment=cfg["symmetry_aug"])

    last_path = os.path.join(CKPT_DIR, f"{tag}_last.pt")
    best_path = os.path.join(CKPT_DIR, f"{tag}_best.pt")
    hist_path = os.path.join(CKPT_DIR, f"{tag}_history.json")

    if cfg.get("init_from") and not (cfg["resume"] and os.path.exists(last_path)):
        ck = torch.load(cfg["init_from"], map_location=device, weights_only=False)
        load_init_weights(policy_net, ck["policy"], n_planes)
        target_net.load_state_dict(policy_net.state_dict())
        print("Khoi tao tu", cfg["init_from"])

    history, best, start_ep, global_step = [], -1.0, 0, 0

    if cfg["resume"] and os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        policy_net.load_state_dict(ck["policy"])
        target_net.load_state_dict(ck["target"])
        opt.load_state_dict(ck["opt"])
        history, best = ck["history"], ck["best"]
        start_ep, global_step = ck["episode"], ck["global_step"]
        print(f"Tiep tuc tu episode {start_ep} (best = {best:.1%}); buffer duoc lam moi")

    run_steps = global_step
    eps = cfg["eps_start"]
    losses, level_counts, opp_counts = [], Counter(), Counter()
    opp_names = list(cfg["opponent_mix"])
    opp_weights = [cfg["opponent_mix"][k] for k in opp_names]
    t0 = time.time()

    for ep in range(start_ep, cfg["num_episodes"]):
        lvl = mix.choices(cfg["train_levels"], weights=cfg["level_weights"])[0]
        opp = mix.choices(opp_names, weights=opp_weights)[0]
        level_counts[lvl] += 1
        opp_counts[opp] += 1
        obs, info = env.reset(level=lvl, opponent=opp)
        mask, done = info["legal_mask"], False

        while not done:
            eps = max(cfg["eps_end"], cfg["eps_start"]
                      - (cfg["eps_start"] - cfg["eps_end"]) * run_steps / cfg["eps_decay_steps"])
            action = select_action(policy_net, env, obs, mask, eps, rng, cfg)
            obs2, r, term, trunc, info = env.step(action)
            done = term or trunc
            buffer.push(obs, action, r, obs2, info["legal_mask"], float(done))
            obs, mask = obs2, info["legal_mask"]
            global_step += 1
            run_steps += 1
            if len(buffer) >= cfg["learn_start"] and global_step % cfg["train_freq"] == 0:
                losses.append(optimize(policy_net, target_net, opt, buffer, cfg))
                soft_update(target_net, policy_net, cfg["tau"])

        if (ep + 1) % 50 == 0:
            print(f"  ep {ep+1} | step {global_step} | {(time.time()-t0)/60:.1f} phut", flush=True)

        if (ep + 1) % cfg["eval_every"] == 0:
            ev2 = evaluate(policy_net, cfg["eval_level"], cfg["eval_games"], cfg["eval_seed"], cfg)
            ev1 = evaluate(policy_net, cfg["extra_eval_level"], cfg["eval_games"], cfg["eval_seed"], cfg)
            score = 0.5 * (ev2["checkmate_rate"] + ev1["checkmate_rate"])
            row = dict(
                episode=ep + 1, global_step=global_step, eps=float(eps),
                loss=float(np.mean(losses[-500:])) if losses else float("nan"),
                score=score,
                mate_L2=ev2["checkmate_rate"], loss_L2=ev2["queen_loss_rate"],
                repeat_L2=ev2["results"].get("fivefold_repetition", 0) / cfg["eval_games"],
                mate_L1=ev1["checkmate_rate"], loss_L1=ev1["queen_loss_rate"],
                repeat_L1=ev1["results"].get("fivefold_repetition", 0) / cfg["eval_games"],
            )
            history.append(row)
            if score > best:
                best = score
                torch.save({"policy": policy_net.state_dict(), "cfg": cfg,
                            "episode": ep + 1, "score": best}, best_path)
            torch.save({"policy": policy_net.state_dict(), "target": target_net.state_dict(),
                        "opt": opt.state_dict(), "history": history, "best": best,
                        "episode": ep + 1, "global_step": global_step, "cfg": cfg}, last_path)
            with open(hist_path, "w") as f:
                json.dump(history, f)
            print(f"ep {ep+1:6d} | eps {eps:.2f} | loss {row['loss']:.4f} | "
                  f"L2 vs defender: mate {row['mate_L2']:.1%} mat hau {row['loss_L2']:.1%} "
                  f"lap {row['repeat_L2']:.1%} | L1: mate {row['mate_L1']:.1%} "
                  f"lap {row['repeat_L1']:.1%} | score {score:.1%} | "
                  f"{(time.time()-t0)/60:.1f} phut", flush=True)

    total = sum(level_counts.values())
    print("Ti le level:", {k: f"{v/total:.0%}" for k, v in sorted(level_counts.items())})
    print("Ti le doi thu:", {k: f"{v/total:.0%}" for k, v in opp_counts.items()})
    return policy_net, history


In [ ]:
smoke_cfg = dict(CONFIG, num_episodes=40, eval_every=20, eval_games=20,
                 learn_start=200, resume=False)
_ = train(smoke_cfg, tag="smokeA")


Mo rong conv1 tu 3 sang 6 kenh (cac kenh them khoi tao bang 0)
Khoi tao tu /content/drive/MyDrive/KHDL_baseline/checkpoints/baseline_L0_best.pt
ep     20 | eps 0.30 | loss 0.1812 | L2 vs defender: mate 15.0% mat hau 45.0% lap 40.0% | L1: mate 10.0% lap 35.0% | score 12.5% | 0.7 phut
ep     40 | eps 0.29 | loss 0.1526 | L2 vs defender: mate 0.0% mat hau 15.0% lap 60.0% | L1: mate 5.0% lap 35.0% | score 2.5% | 2.0 phut
Ti le level: {0: '12%', 1: '28%', 2: '60%'}
Ti le doi thu: {'defender': '72%', 'random': '28%'}


In [ ]:
policy_netA, historyA = train(CONFIG, tag="levelA")


Tiep tuc tu episode 20000 (best = 99.2%); buffer duoc lam moi
Ti le level: {}
Ti le doi thu: {}


In [ ]:
ck = torch.load(os.path.join(CKPT_DIR, "levelA_best.pt"), map_location=device, weights_only=False)
cfg_b = ck["cfg"]
n_planes = kqk.N_PLANES_ATTACK if cfg_b["attack_planes"] else kqk.N_PLANES_BASE

best_net = QNet(n_planes).to(device)
best_net.load_state_dict(ck["policy"])
print(f"Checkpoint episode {ck['episode']} | score tren seed {cfg_b['eval_seed']} = {ck['score']:.1%}")

env_f = kqk.KQKEnv(max_moves=cfg_b["max_moves"], shaping_weight=0.0, attack=cfg_b["attack_planes"])
players = (
    ("random_bot", kqk.random_policy),
    ("greedy_bot", kqk.greedy_policy),
    ("DQN", make_agent_policy(best_net)),
)
rows = []
for level in (0, 1, 2):
    for name, pol in players:
        s = kqk.play_games(env_f, pol, n_games=500, level=level, seed=HOLDOUT_SEED)
        rows.append(dict(
            level=level, player=name,
            checkmate=s["checkmate_rate"], queen_loss=s["queen_loss_rate"],
            stalemate=s["stalemate_rate"], timeout=s["timeout_rate"],
            mean_length=s["mean_length"], mean_reward=s["mean_reward"],
        ))

df = pd.DataFrame(rows)
df.to_csv(os.path.join(PROJECT_DIR, "final_results_A.csv"), index=False)
display(df.style.format({
    "checkmate": "{:.1%}", "queen_loss": "{:.1%}", "stalemate": "{:.1%}",
    "timeout": "{:.1%}", "mean_length": "{:.1f}", "mean_reward": "{:.2f}",
}))


Checkpoint episode 17000 | score tren seed 10001 = 99.2%


,level,player,checkmate,queen_loss,stalemate,timeout,mean_length,mean_reward
0,0,random_bot,1.0%,79.4%,4.6%,15.0%,23.5,-1.17
1,0,greedy_bot,6.4%,74.4%,3.0%,16.2%,23.1,-0.84
2,0,DQN,100.0%,0.0%,0.0%,0.0%,7.6,4.93
3,1,random_bot,2.2%,76.2%,7.4%,14.2%,22.7,-1.09
4,1,greedy_bot,14.6%,66.8%,5.6%,13.0%,20.2,-0.32
5,1,DQN,99.6%,0.2%,0.2%,0.0%,6.3,4.92
6,2,random_bot,1.2%,74.6%,7.0%,17.2%,22.6,-1.14
7,2,greedy_bot,12.2%,65.8%,7.0%,15.0%,22.4,-0.48
8,2,DQN,100.0%,0.0%,0.0%,0.0%,7.0,4.94


In [ ]:
env_def = kqk.KQKEnv(max_moves=cfg_b["max_moves"], shaping_weight=0.0,
                     attack=cfg_b["attack_planes"], opponent="defender")
rows = []
for level in (0, 1, 2):
    s = kqk.play_games(env_def, make_agent_policy(best_net), n_games=300,
                       level=level, seed=HOLDOUT_SEED)
    rows.append(dict(level=level, checkmate=s["checkmate_rate"],
                     queen_loss=s["queen_loss_rate"],
                     stalemate=s["stalemate_rate"], timeout=s["timeout_rate"],
                     mean_length=s["mean_length"]))
display(pd.DataFrame(rows).style.format({
    "checkmate": "{:.1%}", "queen_loss": "{:.1%}", "stalemate": "{:.1%}",
    "timeout": "{:.1%}", "mean_length": "{:.1f}",
}))


,level,checkmate,queen_loss,stalemate,timeout,mean_length
0,0,100.0%,0.0%,0.0%,0.0%,7.0
1,1,96.7%,0.3%,0.0%,0.0%,7.2
2,2,98.7%,0.0%,0.0%,0.0%,8.0


In [ ]:
for level in (0, 1, 2):
    s = kqk.play_games(env_def, make_agent_policy(best_net), n_games=300,
                       level=level, seed=HOLDOUT_SEED)
    print(f"L{level}:", s["results"])


L0: {'checkmate': 300}
L1: {'checkmate': 290, 'fivefold_repetition': 9, 'insufficient_material': 1}
L2: {'checkmate': 296, 'fivefold_repetition': 4}


In [ ]:
def trace_game(net, level=0, seed=HOLDOUT_SEED):
    env = kqk.KQKEnv(max_moves=50, shaping_weight=0.0, attack=True, opponent="defender")
    pol = make_agent_policy(net)
    env.reset(seed=seed, level=level)
    plies, done = [], False
    while not done:
        a = pol(env, None)
        n0 = len(env.board.move_stack)
        _, r, term, trunc, info = env.step(a)
        plies += [m.uci() for m in env.board.move_stack[n0:]]
        done = term or trunc
    print(info["result"], "|", " ".join(plies))

trace_game(best_net, level=0)


checkmate | e1f2 e5e4 d1d6 e4f5 f2f3 f5g5 d6e6 g5h5 f3f4 h5h4 e6g8 h4h3 g8g3
